# Эксперимент 21: Acoustic predictors of pediatric dysarthria

**Статья:** Acoustic Predictors of Pediatric Dysarthria in Cerebral Palsy (Акустические предикторы педиатрической дизартрии при церебральном параличе) 2018

**Ссылка:** [https://pmc.ncbi.nlm.nih.gov/articles/PMC5963041/](https://pmc.ncbi.nlm.nih.gov/articles/PMC5963041/)

**Краткое описание модели:** Расширенный клинический pipeline: vocal-tract признаки + просодические маркеры (speech/pause ratio, onset rate, tempo) -> регуляризованный классификатор.

**Содержание статьи:** Пайплайн воспроизводит логику acoustic predictors: используются интерпретируемые акустические и просодические характеристики, а не ASR-proxy признаки.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, classification_report

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent))

from shared import config, data_utils
from shared.results_utils import save_result_csv

## 1. Загрузка данных и разбиение

In [ ]:
paths_train, paths_val, paths_test, y_train, y_val, y_test, letters_train, letters_val, letters_test = data_utils.get_splits()
print(f"Train: {len(paths_train)}, Val: {len(paths_val)}, Test: {len(paths_test)}")
n_letters = letters_train.shape[1]

Train: 1942, Val: 417, Test: 417


## 2. Подготовка признаков / представлений

In [4]:
def extractor(path):
    print(path)
    vt = data_utils.extract_vocal_tract_features(path)
    y, sr = data_utils.load_audio(path, sr=config.TARGET_SR, max_sec=config.MAX_DURATION_SEC)
    total_sec = max(len(y) / sr, 1e-6)
    intervals = librosa.effects.split(y, top_db=25, frame_length=config.WIN_LENGTH, hop_length=config.HOP_LENGTH)
    speech_sec = float(np.sum((intervals[:, 1] - intervals[:, 0]) / sr)) if len(intervals) else 0.0
    speech_ratio = speech_sec / total_sec
    pause_ratio = max(0.0, 1.0 - speech_ratio)
    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=config.HOP_LENGTH)
    onset_rate = float(np.sum(onset_env > np.percentile(onset_env, 75))) / total_sec if len(onset_env) else 0.0
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr, hop_length=config.HOP_LENGTH)
    return np.concatenate([vt.astype(np.float32), np.array([speech_ratio, pause_ratio, onset_rate, float(tempo)], dtype=np.float32)], axis=0)

X_train = data_utils.build_feature_matrix(paths_train, extractor, n_jobs=-1)
X_val   = data_utils.build_feature_matrix(paths_val, extractor, n_jobs=-1)
X_test  = data_utils.build_feature_matrix(paths_test, extractor, n_jobs=-1)
if letters_train.shape[1] > 0:
    X_train = np.hstack([X_train, letters_train])
    X_val   = np.hstack([X_val, letters_val])
    X_test  = np.hstack([X_test, letters_test])


/mnt/d/Projects/HSE/VKR/VKR/data/bad/eu.29c784bf-b14e-4820-a181-252e0acb15c4.wav
/mnt/d/Projects/HSE/VKR/VKR/data/bad/eu.342652bc-6976-4508-aaa1-3786c2d555b0.wav
/mnt/d/Projects/HSE/VKR/VKR/data/good/eu.3e66832d-b067-4311-b532-41e0206a8a9b.wav
/mnt/d/Projects/HSE/VKR/VKR/data/bad/eu.b43450d4-ec1d-48a1-a40e-5b5000537f4f.wav
/mnt/d/Projects/HSE/VKR/VKR/data/bad/eu.e353090e-1bd3-4b45-9c57-b33f8319cbcf.wav
/mnt/d/Projects/HSE/VKR/VKR/data/good/eu.0ae8e74f-7e5b-4200-bce0-07936d92392c.wav
/mnt/d/Projects/HSE/VKR/VKR/data/bad/eu.b43450d4-ec1d-48a1-a40e-5b5000537f4f.wav


: 

## 3. Обучение, оценка и сохранение результата

In [ ]:
EXPERIMENT_ID = "exp_21_acoustic_predictors"
EXPERIMENT_NAME = "21 - Acoustic predictors (clinical protocol)"
MODEL_NAME = "LogReg (balanced)"

pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(class_weight="balanced", max_iter=3000, solver="liblinear"))])
param_grid = {"clf__C": [0.1, 0.3, 1.0, 3.0, 10.0]}
grid = GridSearchCV(pipe, param_grid, cv=5, scoring="f1_macro", refit=True, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)
clf = grid.best_estimator_

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]
accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_bad = f1_score(y_test, y_pred, average="binary", pos_label=config.CLASS_BAD)
roc_auc = roc_auc_score(y_test, y_proba)
precision_bad = precision_score(y_test, y_pred, zero_division=0, pos_label=config.CLASS_BAD)
recall_bad = recall_score(y_test, y_pred, zero_division=0, pos_label=config.CLASS_BAD)

print(classification_report(y_test, y_pred, target_names=config.CLASS_NAMES))
display(pd.DataFrame([{"accuracy": accuracy, "f1_macro": f1_macro, "f1_bad": f1_bad, "roc_auc": roc_auc, "precision_bad": precision_bad, "recall_bad": recall_bad}]))

save_result_csv(
    exp_dir=exp_dir, 
    experiment_id=EXPERIMENT_ID, 
    experiment_name=EXPERIMENT_NAME, 
    model=MODEL_NAME, 
    accuracy=accuracy, 
    f1_macro=f1_macro, 
    f1_bad=f1_bad, 
    roc_auc=roc_auc, 
    precision_bad=precision_bad, 
    recall_bad=recall_bad, 
    notes=f"acoustic_predictors/grid={grid.best_params_}",
    num_params=None,
    train_time_sec=None
)